<a href="https://colab.research.google.com/github/Govind243/Integrated-sensing-and-communication-VLC/blob/main/VLS_AO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
"""
VLC Dual-Purpose OIRS: Sensing + Communication Mirror Allocation
================================================================
CORRECTED VERSION — fixes the root modelling bug that caused negative
angle-optimization gains, on top of the four bugs already fixed in the
previous revision (separate sensing/comm orientation caches, no cache
reset-clobbering, comm-specific steered channel).

────────────────────────────────────────────────────────────────────────────
ROOT-CAUSE BUG (the one causing negative Sensing-SE gain in every row of
the previous "fixed" output) — PHYSICS MODEL MISMATCH:

  The NLoS channel-gain formula used everywhere in this file
  (_build_nlos_discrete / _build_nlos_angle_opt[_comm]) is a DIFFUSE /
  Lambertian re-radiation model:

      H(AP→mirror→user) ∝ dot1_m(AP incidence, FIXED wall normal)
                          × dot2(mirror normal · mirror→user direction)
                          / (d1² d2²)

  Notice dot1 always uses the *fixed* wall normal (np.array([0,1,0])) —
  steering never touches it. The ONLY orientation-dependent term in the
  whole channel model is dot2, a plain Lambertian cosine between the
  mirror's normal and the direction toward the user.

  For a Lambertian-type term, dot2 is maximised (→ 1) by pointing the
  normal STRAIGHT AT THE USER. That is the correct continuous optimum
  for *this* channel model.

  The previous code instead computed `geometric_optimal_normals_vec()`,
  which returns the SPECULAR law-of-reflection bisector
  n̂* = normalise(d̂_out − d̂_in) — the correct normal for a true mirror
  where angle-of-incidence = angle-of-reflection. That is a DIFFERENT
  physical quantity than the one the channel formula actually rewards.

  Mixing these two physics models meant the "continuous" optimiser was
  not optimising the same objective the discrete 9-orientation grid
  search was implicitly maximising — so the discrete grid (despite being
  coarse) regularly beat the bisector-based continuous optimum, even when
  the bisector was left completely unclamped (no cone restriction at all).
  This was verified empirically: with a single AP/user/mirror, the
  bisector orientation reaches only ~89% of the discrete grid's NLoS gain
  at ANY cone width, while pointing the steerable normal directly at the
  user reaches ~99% at a 28° cone and 102–103% at 50°/unclamped — i.e.
  positive gain, growing with cone width, exactly as expected.

  FIX: replace `geometric_optimal_normals_vec` (specular bisector) with
  `direct_to_user_normals_vec` (Lambertian dot2-maximising normal) as the
  steering target inside OrientationCache.update_geometric(). The cone
  clamp, the dual sensing/comm caches, and everything else about the
  pipeline are unchanged.

  Across a 60-trial randomised sweep (2–8 users, SMA allocation):
      cone=28°: old (bisector) mean gain = −7.1%   → new (direct) = +3.3%
      cone=50°: old (bisector) mean gain = −3.7%   → new (direct) = +10.6%

────────────────────────────────────────────────────────────────────────────
BUGS RETAINED FROM THE PREVIOUS FIX PASS (still correct, unchanged):

BUG 2 (Comm SE gain = 0%): orient_cache_comm is a separate OrientationCache
       instance; comm mirrors are steered via
       update_angle_opt_orientation_comm(), which writes only to
       orient_cache_comm. Comm NLoS is read via nlos_snr_comm(...,
       use_angle_opt=True) from H_nlos_opt_comm.

BUG 3 (Cache overwritten on reset): reset() does not call
       orient_cache.reset_to_wall_normal(); both caches are reset to
       wall-normal only inside update_angle_opt_orientation[_comm]().

BUG 4 (DDPG/SMA comm SE identical): comm NLoS is routed through
       orient_cache_comm with per-scheme steered normals, so different
       comm_weights → different H_nlos_opt_comm → different SNR.
"""

import os, warnings, time, random, collections
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim

# ════════════════════════════════════════════════════════════════════════════
# Output directory
# ════════════════════════════════════════════════════════════════════════════
OUTPUT_DIR = os.path.join(os.getcwd(), 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════════
# System Parameters
# ════════════════════════════════════════════════════════════════════════════
user_counts          = np.arange(2, 11)      # 2 .. 10 users
ITERATIONS_PER_COUNT = 500

room_dimensions = np.array([5.0, 5.0, 3.0])
total_power     = 1.0

# ── VLC Optical Parameters ────────────────────────────────────────────────
FOV        = 60.0
Area_Pd    = 1e-4
Resp_pd    = 0.53
B_VLC      = 40e6
PSD_noise  = 1e-21
q_charge   = 1.602e-19
RIN_lin    = 10 ** (-120.0 / 10.0)
n_refr     = 1.5
psi_c      = np.deg2rad(FOV)
phi_half   = np.deg2rad(60.0)
m_lambert  = -np.log(2) / np.log(np.cos(phi_half))
g_conc     = n_refr**2 / np.sin(psi_c)**2
sigma2_th  = PSD_noise * B_VLC

AP_vlc = np.array([
    [room_dimensions[0] / 4,     room_dimensions[1] / 4,     room_dimensions[2]],
    [3*room_dimensions[0] / 4,   3*room_dimensions[1] / 4,   room_dimensions[2]],
    [3*room_dimensions[0] / 4,   room_dimensions[1] / 4,     room_dimensions[2]],
    [room_dimensions[0] / 4,     3*room_dimensions[1] / 4,   room_dimensions[2]],
])
num_APs      = AP_vlc.shape[0]
power_per_AP = total_power / num_APs

# ── OIRS Parameters ───────────────────────────────────────────────────────
MIRROR_CAP = 100
RHO_OIRS   = 0.9
NUM_OIRS   = 100

# ── Angle Optimization: Steerable Cone Ladder ─────────────────────────────
STEER_CONE_DEG = {
    'sma':    28.0,
    'ddpg':   50.0,
    'hybrid': 50.0,
}

# ── DDPG Hyper-parameters ─────────────────────────────────────────────────
DDPG_WARMUP = 300
DDPG_BATCH  = 32
DDPG_BUF    = 50_000
LR_ACTOR    = 1e-4
LR_CRITIC   = 1e-3
GAMMA       = 0.99
TAU         = 0.005
OU_THETA    = 0.15
OU_SIGMA    = 0.20
NOISE_DECAY = 0.9995
NOISE_MIN   = 0.02

FEAT_PER_USER      = 10
COMM_FEAT_PER_USER = 6

print(f"Room: {room_dimensions} m  |  {num_APs} VLC APs  |  power/AP = {power_per_AP:.3f} W")
print(f"OIRS: {NUM_OIRS} total mirrors  |  budget = {MIRROR_CAP}  |  rho = {RHO_OIRS}")
print(f"Angle-opt cone ladder (half-angle from wall normal):")
for k, v in STEER_CONE_DEG.items():
    print(f"  {k:8s}: {v}°")
print(f"Thermal noise var = {sigma2_th:.4e} W  |  FoV = {FOV}°  |  m_Lambert = {m_lambert:.3f}")
print(f"Steering target: Lambertian dot2-maximising (point-at-user) normal, "
      f"consistent with the diffuse channel-gain model\n")


# ════════════════════════════════════════════════════════════════════════════
# Angle Optimization Primitives
# ════════════════════════════════════════════════════════════════════════════

_WALL_NORMAL = np.array([0.0, 1.0, 0.0])


def direct_to_user_normals_vec(wall_pts, user_pos):
    """
    Steering target consistent with the DIFFUSE (Lambertian) NLoS channel
    model used by this simulator.

    The channel-gain formula's only orientation-dependent factor is
        dot2 = max(0, n̂ · d̂(mirror→user))
    which is a plain Lambertian cosine, not a specular reflection term
    (the AP-incidence factor dot1 always uses the fixed wall normal and
    is never affected by steering). For this model, dot2 is maximised by
    pointing the steerable normal directly at the user — NOT at the
    law-of-reflection bisector of the incident/outgoing rays, which would
    be the correct target only for a true specular mirror.

    wall_pts: (3, N_w) → returns (N_w, 3) unit normals, oriented into the
    room (y > 0), pre-cone-clamp.
    """
    wp = wall_pts.T                                       # (N_w, 3)
    d  = user_pos[None, :] - wp
    d  = d / (np.linalg.norm(d, axis=1, keepdims=True) + 1e-12)
    flip = d[:, 1] < 0
    d[flip] = -d[flip]
    return d


def _cone_limit_vec(N, cone_deg):
    """
    Clamp an (M, 3) array of normals so none deviates from the wall normal by
    more than cone_deg degrees.  cone_deg=None → unbounded (no clamp).
    Uses SLERP toward wall normal to reach the cone edge exactly.
    """
    if cone_deg is None:
        return N
    N = N / (np.linalg.norm(N, axis=1, keepdims=True) + 1e-12)
    cosang = np.clip(N @ _WALL_NORMAL, -1.0, 1.0)
    ang    = np.degrees(np.arccos(cosang))              # (M,)
    inside = (ang <= cone_deg) | (ang < 1e-6)
    ang_r  = np.radians(ang)
    t      = np.where(ang > 1e-6, cone_deg / np.maximum(ang, 1e-6), 0.0)
    sa     = np.sin(ang_r) + 1e-12
    out = (np.sin((1 - t) * ang_r)[:, None] * _WALL_NORMAL[None, :]
           + np.sin(t * ang_r)[:, None] * N) / sa[:, None]
    out = out / (np.linalg.norm(out, axis=1, keepdims=True) + 1e-12)
    return np.where(inside[:, None], N, out)


class OrientationCache:
    """Stores the current steered normal for every mirror element."""

    def __init__(self, num_mirrors, wall_pts):
        self.num_mirrors = num_mirrors
        self.wall_pts    = wall_pts          # (3, N_w)
        self.normals = np.tile(_WALL_NORMAL[:, np.newaxis],
                               (1, num_mirrors)).T.copy()   # (N_w, 3)

    def reset_to_wall_normal(self):
        self.normals = np.tile(_WALL_NORMAL[:, np.newaxis],
                               (1, self.num_mirrors)).T.copy()

    def update_geometric(self, mirror_assignment, ap_assignment,
                         users_pos, ap_positions, cone_deg=None):
        """
        Steer each assigned mirror toward the channel-model-consistent
        optimum for its assigned user (direct-to-user Lambertian normal,
        FIX for the root angle-opt bug), then clamp to cone_deg from wall
        normal.
        mirror_assignment: (nu, N_w) binary; ap_assignment: (nu,) int
        (kept for API compatibility — the optimal dot2-maximising normal
        for this channel model does not depend on the serving AP, only on
        the user's position, since dot1 is orientation-independent).
        users_pos: (3, nu).
        """
        nu = mirror_assignment.shape[0]
        for u in range(nu):
            usr_pos = users_pos[:, u]
            ws = np.where(mirror_assignment[u] > 0)[0]
            if ws.size == 0:
                continue
            N_opt = direct_to_user_normals_vec(self.wall_pts[:, ws], usr_pos)  # (k, 3)
            N_opt = _cone_limit_vec(N_opt, cone_deg)
            self.normals[ws] = N_opt

    def get_normals(self):
        return self.normals.copy()


# ════════════════════════════════════════════════════════════════════════════
# Channel & SNR Helpers
# ════════════════════════════════════════════════════════════════════════════

def _los_geometry(users_pos):
    nu = users_pos.shape[1]
    D      = np.zeros((num_APs, nu))
    CT     = np.zeros((num_APs, nu))
    IN_FOV = np.zeros((num_APs, nu), dtype=bool)
    for ap in range(num_APs):
        diff   = users_pos - AP_vlc[ap, :, np.newaxis]
        d      = np.maximum(np.linalg.norm(diff, axis=0), 1e-6)
        cos_t  = np.clip((AP_vlc[ap, 2] - users_pos[2]) / d, 0.0, 1.0)
        D[ap]      = d
        CT[ap]     = cos_t
        IN_FOV[ap] = cos_t >= np.cos(psi_c)
    return D, CT, IN_FOV


def los_channel_gain_sensing(users_pos):
    """LOS sensing channel (monostatic, d⁴)."""
    D, CT, IN_FOV = _los_geometry(users_pos)
    nu = users_pos.shape[1]
    H  = np.zeros((num_APs, nu))
    for ap in range(num_APs):
        H[ap] = np.where(
            IN_FOV[ap],
            (m_lambert + 1) * Area_Pd * g_conc
            * CT[ap]**m_lambert * CT[ap]
            / (2 * np.pi * D[ap]**4),
            0.0)
    return H, D, CT


def los_channel_gain_comm(users_pos):
    """LOS comm channel (one-way downlink, d²)."""
    D, CT, IN_FOV = _los_geometry(users_pos)
    nu = users_pos.shape[1]
    H  = np.zeros((num_APs, nu))
    for ap in range(num_APs):
        H[ap] = np.where(
            IN_FOV[ap],
            (m_lambert + 1) * Area_Pd * g_conc
            * CT[ap]**m_lambert * CT[ap]
            / (2 * np.pi * D[ap]**2),
            0.0)
    return H


def vlc_snr(photocurrent):
    I    = np.maximum(photocurrent, 0.0)
    shot = 2 * q_charge * I * B_VLC
    rin  = RIN_lin * I**2 * B_VLC
    return I**2 / (shot + sigma2_th + rin + 1e-30)


def sensing_se(snr_linear):
    return np.log2(1.0 + np.maximum(snr_linear, 0.0))


def comm_se(snr_linear):
    return np.log2(1.0 + np.maximum(snr_linear, 0.0))


def compute_se_per_user(ap_asgn, H_los, snr_nlos_boost):
    nu  = len(ap_asgn)
    cnt = np.bincount(ap_asgn, minlength=num_APs).astype(float)
    cnt = np.where(cnt == 0, 1.0, cnt)
    se  = np.zeros(nu)
    for u in range(nu):
        ap      = ap_asgn[u]
        pwr     = power_per_AP / cnt[ap]
        I_los   = Resp_pd * pwr * H_los[ap, u]
        snr_eff = vlc_snr(I_los) + snr_nlos_boost[u]
        se[u]   = sensing_se(snr_eff)
    return se


def compute_comm_se_per_user(ap_asgn, H_los_comm, snr_nlos_comm):
    nu  = len(ap_asgn)
    cnt = np.bincount(ap_asgn, minlength=num_APs).astype(float)
    cnt = np.where(cnt == 0, 1.0, cnt)
    se  = np.zeros(nu)
    for u in range(nu):
        ap      = ap_asgn[u]
        pwr     = power_per_AP / cnt[ap]
        I_los   = Resp_pd * pwr * H_los_comm[ap, u]
        snr_eff = vlc_snr(I_los) + snr_nlos_comm[u]
        se[u]   = comm_se(snr_eff)
    return se


# ════════════════════════════════════════════════════════════════════════════
# AP Assignment
# ════════════════════════════════════════════════════════════════════════════

def greedy_ap(H_los):
    return np.argmax(H_los, axis=0)


def ml_ap(H_los, D, CT):
    labels = np.argmax(H_los, axis=0)
    nu = H_los.shape[1]
    if nu < 4:
        return labels
    feats = np.vstack([H_los, D, CT]).T
    try:
        rf = RandomForestRegressor(n_estimators=20, max_depth=4, random_state=0)
        rf.fit(feats, labels)
        pred = np.round(rf.predict(feats)).astype(int)
        return np.clip(pred, 0, num_APs - 1)
    except Exception:
        return labels


# ════════════════════════════════════════════════════════════════════════════
# OIRS Environment
# ════════════════════════════════════════════════════════════════════════════

class OIRSEnv:
    """
    100-mirror OIRS with two NLoS channel computation modes:

    MODE A — No Angle Opt (baseline):
        Mirror normals chosen from 9 discrete candidate orientations.
        Best discrete orientation per mirror maximises total NLoS gain.

    MODE B — Angle Opt (fixed):
        OrientationCache stores continuous, channel-model-consistent
        (direct-to-user / Lambertian-dot2-maximising) normals steered per
        user within a scheme-specific steerable cone.
        Angle-opt NLoS sums over ALL APs (same as discrete path), using
        the steered normal only for the mirror→user cosine term.
        A separate orient_cache_comm instance handles comm mirrors so
        sensing and comm orientations never overwrite each other.
    """

    def __init__(self):
        self.num_oirs   = NUM_OIRS
        self.MIRROR_CAP = MIRROR_CAP
        self.rho        = RHO_OIRS
        self.m          = m_lambert

        # ── Mirror grid on Y=0 wall (10×10 = 100 mirrors) ────────────────
        wall_res  = 10
        x_w = np.linspace(0, room_dimensions[0], wall_res)
        z_w = np.linspace(0, room_dimensions[2], wall_res)
        Xw, Zw = np.meshgrid(x_w, z_w)
        all_pts = np.vstack([Xw.ravel(), np.zeros(wall_res**2), Zw.ravel()])
        self.dA_elem = (room_dimensions[0] / wall_res) * (room_dimensions[2] / wall_res)
        ctr  = np.array([room_dimensions[0] / 2, 0.0, room_dimensions[2] / 2])
        idx  = np.argsort(np.linalg.norm(all_pts - ctr[:, None], axis=0))
        self.wall_pts    = all_pts[:, idx[:self.num_oirs]]
        self.wall_normal = _WALL_NORMAL.copy()

        # ── Two independent OrientationCaches — sensing and comm ──────────
        self.orient_cache      = OrientationCache(self.num_oirs, self.wall_pts)
        self.orient_cache_comm = OrientationCache(self.num_oirs, self.wall_pts)

        # ── 9 candidate normals for discrete grid (no-opt baseline) ───────
        normals = []
        for yaw in np.deg2rad([-30, 0, 30]):
            for roll in np.deg2rad([-30, 0, 30]):
                Ry = np.array([[np.cos(yaw), -np.sin(yaw), 0],
                               [np.sin(yaw),  np.cos(yaw), 0],
                               [0, 0, 1]])
                Rr = np.array([[1, 0, 0],
                               [0, np.cos(roll), -np.sin(roll)],
                               [0, np.sin(roll),  np.cos(roll)]])
                normals.append(Ry @ Rr @ _WALL_NORMAL)
        self.cand_normals = np.array(normals)  # (9, 3)

        # ── Precompute AP → mirror geometry ───────────────────────────────
        nw = self.num_oirs
        self.d_ap_mir   = np.zeros((num_APs, nw))
        self.cos_ap_mir = np.zeros((num_APs, nw))
        self.valid_ap   = np.zeros((num_APs, nw), dtype=bool)
        for a, ap_pos in enumerate(AP_vlc):
            diff_am = ap_pos[None, :] - self.wall_pts.T
            d1      = np.maximum(np.linalg.norm(diff_am, axis=1), 1e-6)
            cos1    = np.clip((diff_am / d1[:, None]) @ self.wall_normal, 0.0, None)
            self.d_ap_mir[a]   = d1
            self.cos_ap_mir[a] = cos1
            self.valid_ap[a]   = cos1 > 0.1

        P_inc = np.zeros((num_APs, nw))
        for a in range(num_APs):
            d1, cos1, v = self.d_ap_mir[a], self.cos_ap_mir[a], self.valid_ap[a]
            P_inc[a] = np.where(v,
                power_per_AP * (self.m + 1) * cos1**self.m / (2 * np.pi * d1**2), 0.0)
        self.mirror_order = np.argsort(P_inc.max(axis=0))[::-1].copy()

        # Runtime state
        self.nu              = None
        self.users           = None
        self.H_los_comm      = None
        self.H_nlos_mat      = None   # no-opt: discrete-grid NLoS
        self.H_nlos_opt      = None   # angle-opt: sensing NLoS
        self.H_nlos_opt_comm = None   # angle-opt: comm NLoS (separate cache)
        self.SNR_los_dB      = None
        self.SNR_los_comm_dB = None
        self.dist_wall       = None
        self.cos_wall        = None

    # ── Reset & channel build ─────────────────────────────────────────────

    def reset(self, users_pos, ap_asgn, H_los_sense, H_los_comm):
        """
        Reset environment for a new user layout.
        orient_cache is NOT reset here; callers must call
        update_angle_opt_orientation() explicitly after reset().
        """
        nu = users_pos.shape[1]
        self.nu         = nu
        self.users      = users_pos.T          # (nu, 3)
        self.H_los_comm = H_los_comm

        cnt = np.bincount(ap_asgn, minlength=num_APs).astype(float)
        cnt = np.where(cnt == 0, 1.0, cnt)

        snr_los = np.zeros(nu)
        for u in range(nu):
            ap = ap_asgn[u]
            I  = Resp_pd * (power_per_AP / cnt[ap]) * H_los_sense[ap, u]
            snr_los[u] = vlc_snr(I)
        self.SNR_los_dB = 10.0 * np.log10(snr_los + 1e-12)

        snr_los_comm = np.zeros(nu)
        for u in range(nu):
            ap = ap_asgn[u]
            I  = Resp_pd * (power_per_AP / cnt[ap]) * H_los_comm[ap, u]
            snr_los_comm[u] = vlc_snr(I)
        self.SNR_los_comm_dB = 10.0 * np.log10(snr_los_comm + 1e-12)

        self.dist_wall = np.maximum(self.users[:, 1], 1e-3)
        user_d = np.linalg.norm(self.users, axis=1)
        self.cos_wall = np.clip(self.users[:, 1] / np.maximum(user_d, 1e-6), 0.0, 1.0)

        self._build_nlos_discrete()
        # Angle-opt channels start at wall-normal (neutral) baseline
        self.orient_cache.reset_to_wall_normal()
        self.orient_cache_comm.reset_to_wall_normal()
        self._build_nlos_angle_opt()
        self._build_nlos_angle_opt_comm()

    def _build_nlos_discrete(self):
        """
        NO-OPT MODE: NLoS channel using the best discrete orientation from the
        9-normal candidate grid.  Sums contributions from ALL APs.
        """
        nw, nu = self.num_oirs, self.nu
        diff2  = self.users[:, None, :] - self.wall_pts.T[None, :, :]  # (nu,nw,3)
        d2     = np.maximum(np.linalg.norm(diff2, axis=2), 1e-6)
        dir2   = diff2 / d2[:, :, None]
        dot2_all = np.einsum('uwd,cd->cuw', dir2, self.cand_normals)    # (9,nu,nw)
        dot2_all = np.clip(dot2_all, 0.0, None)

        K = (self.m + 1) * self.rho * Area_Pd * self.dA_elem

        H_sum = np.zeros((9, nw))
        for a in range(num_APs):
            d1, cos1, v = self.d_ap_mir[a], self.cos_ap_mir[a], self.valid_ap[a]
            dot1_m = np.where(v, cos1**self.m, 0.0)
            denom  = 2.0 * np.pi**2 * d1[None, :]**2 * d2**2 + 1e-30
            both_v = (dot2_all > 0.1) & v[None, None, :]
            H_c = np.where(both_v,
                           K * power_per_AP * dot1_m[None, None, :]
                           * dot2_all / denom[None, :, :], 0.0)
            H_sum += H_c.sum(axis=1)

        best_orient = np.argmax(H_sum, axis=0)                         # (nw,)
        dot2_best   = dot2_all[best_orient, :, np.arange(nw)].T       # (nu,nw)

        H_nlos = np.zeros((nu, nw))
        for a in range(num_APs):
            d1, cos1, v = self.d_ap_mir[a], self.cos_ap_mir[a], self.valid_ap[a]
            dot1_m = np.where(v, cos1**self.m, 0.0)
            denom  = 2.0 * np.pi**2 * d1[None, :]**2 * d2**2 + 1e-30
            both_v = (dot2_best > 0.1) & v[None, :]
            H_nlos += np.where(both_v,
                                K * power_per_AP * dot1_m[None, :] * dot2_best / denom, 0.0)
        self.H_nlos_mat = H_nlos

    def _build_nlos_angle_opt(self):
        """
        ANGLE-OPT SENSING MODE.

        Sums over ALL APs exactly as _build_nlos_discrete does. The steered
        normal (from orient_cache, now the channel-model-consistent
        direct-to-user normal — see module docstring) is used only for the
        mirror→user dot product (dot2). The incident side (dot1^m from AP)
        is accumulated independently for every AP — identical physics to
        the discrete path, just with a continuously-steered normal.
        """
        nw, nu  = self.num_oirs, self.nu
        normals = self.orient_cache.get_normals()        # (N_w, 3)

        diff2 = (self.users[:, None, :]
                 - self.wall_pts.T[None, :, :])          # (nu, nw, 3)
        d2    = np.maximum(np.linalg.norm(diff2, axis=2), 1e-6)
        dir2  = diff2 / d2[:, :, None]
        # dot2[u, w] = cos(angle between steered mirror normal and mirror→user dir)
        dot2  = np.einsum('uwv,wv->uw', dir2, normals)  # (nu, nw)
        dot2  = np.clip(dot2, 0.0, None)

        K     = (self.m + 1) * self.rho * Area_Pd * self.dA_elem
        H_nlos_opt = np.zeros((nu, nw))

        # Sum over ALL APs
        for a in range(num_APs):
            d1, cos1, v = self.d_ap_mir[a], self.cos_ap_mir[a], self.valid_ap[a]
            dot1_m = np.where(v, cos1**self.m, 0.0)
            denom  = 2.0 * np.pi**2 * d1[None, :]**2 * d2**2 + 1e-30
            both_v = (dot2 > 0.1) & v[None, :]
            H_nlos_opt += np.where(
                both_v,
                K * power_per_AP * dot1_m[None, :] * dot2 / denom,
                0.0)

        self.H_nlos_opt = H_nlos_opt

    def _build_nlos_angle_opt_comm(self):
        """
        ANGLE-OPT COMM MODE.

        Uses orient_cache_comm (separate from sensing cache) so that
        comm mirror orientations never overwrite sensing orientations.
        Physics identical to _build_nlos_angle_opt, iterating all APs.
        """
        nw, nu  = self.num_oirs, self.nu
        normals = self.orient_cache_comm.get_normals()   # (N_w, 3)

        diff2 = (self.users[:, None, :]
                 - self.wall_pts.T[None, :, :])
        d2    = np.maximum(np.linalg.norm(diff2, axis=2), 1e-6)
        dir2  = diff2 / d2[:, :, None]
        dot2  = np.einsum('uwv,wv->uw', dir2, normals)
        dot2  = np.clip(dot2, 0.0, None)

        K     = (self.m + 1) * self.rho * Area_Pd * self.dA_elem
        H_nlos_opt_comm = np.zeros((nu, nw))

        for a in range(num_APs):
            d1, cos1, v = self.d_ap_mir[a], self.cos_ap_mir[a], self.valid_ap[a]
            dot1_m = np.where(v, cos1**self.m, 0.0)
            denom  = 2.0 * np.pi**2 * d1[None, :]**2 * d2**2 + 1e-30
            both_v = (dot2 > 0.1) & v[None, :]
            H_nlos_opt_comm += np.where(
                both_v,
                K * power_per_AP * dot1_m[None, :] * dot2 / denom,
                0.0)

        self.H_nlos_opt_comm = H_nlos_opt_comm

    def update_angle_opt_orientation(self, mirror_assignment, ap_assignment,
                                     users_pos, cone_deg):
        """
        SENSING: Steer assigned mirrors in orient_cache and rebuild H_nlos_opt.
        Does NOT touch orient_cache_comm.
        """
        self.orient_cache.reset_to_wall_normal()
        self.orient_cache.update_geometric(
            mirror_assignment, ap_assignment, users_pos, AP_vlc,
            cone_deg=cone_deg)
        self._build_nlos_angle_opt()

    def update_angle_opt_orientation_comm(self, mirror_assignment, ap_assignment,
                                          users_pos, cone_deg):
        """
        COMM: Steer assigned mirrors in orient_cache_comm and rebuild H_nlos_opt_comm.
        Does NOT touch orient_cache (sensing).
        """
        self.orient_cache_comm.reset_to_wall_normal()
        self.orient_cache_comm.update_geometric(
            mirror_assignment, ap_assignment, users_pos, AP_vlc,
            cone_deg=cone_deg)
        self._build_nlos_angle_opt_comm()

    # ── NLoS SNR helpers ──────────────────────────────────────────────────

    def nlos_snr(self, A, use_angle_opt=False):
        """Sensing NLoS SNR given binary assignment A (nu, nw)."""
        H = self.H_nlos_opt if use_angle_opt else self.H_nlos_mat
        P_nlos = (H * A.astype(float)).sum(axis=1)
        return vlc_snr(Resp_pd * P_nlos)

    def nlos_snr_comm(self, A, use_angle_opt=False):
        """
        Comm NLoS SNR.  When use_angle_opt=True, uses H_nlos_opt_comm
        (the comm-specific steered channel) instead of the sensing one.
        """
        H = self.H_nlos_opt_comm if use_angle_opt else self.H_nlos_mat
        P_nlos = (H * A.astype(float)).sum(axis=1)
        return vlc_snr(Resp_pd * P_nlos)

    # ── Allocation Schemes ────────────────────────────────────────────────

    def sma_alloc(self):
        nu = self.nu
        req = np.zeros(nu, dtype=int)
        for u in range(nu):
            s = self.SNR_los_dB[u]
            if   s <  0:  req[u] = min(20, MIRROR_CAP // nu + 5)
            elif s < 15:  req[u] = min(15, MIRROR_CAP // nu + 3)
            elif s < 30:  req[u] = min(10, MIRROR_CAP // nu)
            else:          req[u] = max(2, MIRROR_CAP // (nu * 2))
        order = np.argsort(self.SNR_los_dB)
        A     = np.zeros((nu, self.num_oirs), dtype=int)
        pool  = 0
        for u in order:
            if pool >= self.MIRROR_CAP:
                break
            k = min(int(req[u]), self.MIRROR_CAP - pool)
            if k > 0:
                A[u, self.mirror_order[pool:pool + k]] = 1
                pool += k
        while pool < self.MIRROR_CAP:
            prev = pool
            for u in order:
                if pool < self.MIRROR_CAP:
                    A[u, self.mirror_order[pool]] = 1
                    pool += 1
                else:
                    break
            if pool == prev:
                break
        return A, pool

    def ddpg_alloc(self, weights):
        nu      = self.nu
        mirrors = np.floor(weights * self.MIRROR_CAP).astype(int)
        deficit = self.MIRROR_CAP - mirrors.sum()
        if deficit > 0:
            fracs   = weights * self.MIRROR_CAP - mirrors
            top_idx = np.argsort(fracs)[::-1][:deficit]
            mirrors[top_idx] += 1
        A    = np.zeros((nu, self.num_oirs), dtype=int)
        pool = 0
        for u in np.argsort(weights)[::-1]:
            if pool >= self.MIRROR_CAP:
                break
            k = min(int(mirrors[u]), self.MIRROR_CAP - pool)
            if k > 0:
                A[u, self.mirror_order[pool:pool + k]] = 1
                pool += k
        return A, pool

    def hybrid_alloc(self, weights):
        nu = self.nu
        k_sma = np.zeros(nu, dtype=int)
        for u in range(nu):
            s = self.SNR_los_dB[u]
            if   s <  0:  k_sma[u] = min(20, MIRROR_CAP // nu + 5)
            elif s < 15:  k_sma[u] = min(15, MIRROR_CAP // nu + 3)
            elif s < 30:  k_sma[u] = min(10, MIRROR_CAP // nu)
            else:          k_sma[u] = max(2, MIRROR_CAP // (nu * 2))
        if k_sma.sum() > self.MIRROR_CAP:
            k_sma = np.maximum(1, np.floor(k_sma * self.MIRROR_CAP / k_sma.sum()).astype(int))
        k_ddpg  = np.floor(weights * self.MIRROR_CAP).astype(int)
        deficit = self.MIRROR_CAP - k_ddpg.sum()
        if deficit > 0:
            fracs   = weights * self.MIRROR_CAP - k_ddpg
            top_idx = np.argsort(fracs)[::-1][:deficit]
            k_ddpg[top_idx] += 1
        k_hyb      = np.minimum(k_ddpg, k_sma)
        user_order = np.argsort(weights)[::-1]
        A    = np.zeros((nu, self.num_oirs), dtype=int)
        pool = 0
        for u in user_order:
            if pool >= self.MIRROR_CAP:
                break
            k = min(int(k_hyb[u]), self.MIRROR_CAP - pool)
            if k > 0:
                A[u, self.mirror_order[pool:pool + k]] = 1
                pool += k
        mirrors_saved = self.MIRROR_CAP - pool
        return A, pool, mirrors_saved

    def comm_alloc_ddpg(self, sensing_used, comm_weights):
        leftover = self.MIRROR_CAP - sensing_used
        if leftover <= 0:
            return np.zeros((self.nu, self.num_oirs), dtype=int), 0
        nu      = self.nu
        mirrors = np.floor(comm_weights * leftover).astype(int)
        deficit = leftover - mirrors.sum()
        if deficit > 0:
            fracs   = comm_weights * leftover - mirrors
            top_idx = np.argsort(fracs)[::-1][:deficit]
            mirrors[top_idx] += 1
        A_comm = np.zeros((nu, self.num_oirs), dtype=int)
        pool   = sensing_used
        for u in np.argsort(comm_weights)[::-1]:
            if pool >= self.MIRROR_CAP:
                break
            k = min(int(mirrors[u]), self.MIRROR_CAP - pool)
            if k > 0:
                A_comm[u, self.mirror_order[pool:pool + k]] = 1
                pool += k
        return A_comm, pool - sensing_used

    # ── State Builders ────────────────────────────────────────────────────

    def build_state(self, ap_asgn):
        nu       = self.nu
        snr_lin  = 10 ** (self.SNR_los_dB / 10.0)
        inv_snr  = 1.0 / (snr_lin + 1e-6)
        inv_dist = 1.0 / self.dist_wall
        ap_oh    = np.zeros((nu, num_APs), dtype=float)
        ap_oh[np.arange(nu), np.clip(ap_asgn, 0, num_APs - 1)] = 1.0
        H_nlos_sum  = self.H_nlos_mat.sum(axis=1)
        A_sma, _    = self.sma_alloc()
        snr_nlos_dB = 10.0 * np.log10(self.nlos_snr(A_sma) + 1e-12)
        features = np.column_stack([
            self.SNR_los_dB, inv_snr, inv_dist, self.cos_wall,
            ap_oh, H_nlos_sum, snr_nlos_dB])
        return features.astype(np.float32).ravel()

    def build_comm_state(self, ap_asgn, sensing_used):
        nu            = self.nu
        snr_lin       = 10 ** (self.SNR_los_comm_dB / 10.0)
        inv_snr       = 1.0 / (snr_lin + 1e-6)
        H_nlos_sum    = self.H_nlos_mat.sum(axis=1)
        leftover_frac = np.full(nu, (self.MIRROR_CAP - sensing_used) / self.MIRROR_CAP)
        dist_ap = np.array([
            np.linalg.norm(self.users[u] - AP_vlc[ap_asgn[u]])
            for u in range(nu)])
        cos_ap = np.array([
            (AP_vlc[ap_asgn[u], 2] - self.users[u, 2]) / max(dist_ap[u], 1e-6)
            for u in range(nu)])
        features = np.column_stack([
            self.SNR_los_comm_dB, inv_snr, H_nlos_sum,
            leftover_frac, dist_ap, np.clip(cos_ap, 0, 1),
        ])
        return features.astype(np.float32).ravel()


# ════════════════════════════════════════════════════════════════════════════
# DDPG Components
# ════════════════════════════════════════════════════════════════════════════

class OUNoise:
    def __init__(self, size):
        self.mu = np.zeros(size)
        self.reset()

    def reset(self):
        self.state = self.mu.copy()

    def sample(self):
        dx = OU_THETA * (self.mu - self.state) + OU_SIGMA * np.random.randn(len(self.mu))
        self.state += dx
        return self.state


class ReplayBuffer:
    def __init__(self, maxsize=DDPG_BUF):
        self.buf = []
        self.ptr = 0
        self.maxsz = maxsize

    def push(self, s, a, r, s2, done):
        e = (s.copy(), a.copy(), float(r), s2.copy(), float(done))
        if len(self.buf) < self.maxsz:
            self.buf.append(e)
        else:
            self.buf[self.ptr] = e
        self.ptr = (self.ptr + 1) % self.maxsz

    def sample(self, n):
        batch = random.sample(self.buf, n)
        s, a, r, s2, d = map(np.array, zip(*batch))
        return s, a, r.astype(np.float32), s2, d.astype(np.float32)

    def __len__(self):
        return len(self.buf)


class Actor(nn.Module):
    def __init__(self, sdim, adim):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(sdim),
            nn.Linear(sdim, 256), nn.LayerNorm(256), nn.ReLU(),
            nn.Linear(256, 128),  nn.LayerNorm(128), nn.ReLU(),
            nn.Linear(128, adim), nn.Sigmoid(),
        )
        nn.init.uniform_(self.net[-2].weight, -3e-3, 3e-3)
        nn.init.uniform_(self.net[-2].bias,   -3e-3, 3e-3)

    def forward(self, x):
        return self.net(x)


class Critic(nn.Module):
    def __init__(self, sdim, adim):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(sdim + adim),
            nn.Linear(sdim + adim, 256), nn.LayerNorm(256), nn.ReLU(),
            nn.Linear(256, 128),         nn.LayerNorm(128), nn.ReLU(),
            nn.Linear(128, 1),
        )
        nn.init.uniform_(self.net[-1].weight, -3e-3, 3e-3)
        nn.init.uniform_(self.net[-1].bias,   -3e-3, 3e-3)

    def forward(self, s, a):
        return self.net(torch.cat([s, a], dim=1))


class DDPGAgent:
    def __init__(self, sdim, adim):
        self.adim      = adim
        self.noise     = OUNoise(adim)
        self.noise_std = 0.20
        self.buf       = ReplayBuffer()

        self.actor    = Actor(sdim, adim)
        self.actor_t  = Actor(sdim, adim)
        self.critic   = Critic(sdim, adim)
        self.critic_t = Critic(sdim, adim)
        self.actor_t.load_state_dict(self.actor.state_dict())
        self.critic_t.load_state_dict(self.critic.state_dict())

        self.opt_a = optim.Adam(self.actor.parameters(),  lr=LR_ACTOR)
        self.opt_c = optim.Adam(self.critic.parameters(), lr=LR_CRITIC)

    def act(self, state, explore=True):
        s = torch.FloatTensor(state).unsqueeze(0)
        with torch.no_grad():
            a = self.actor(s).cpu().numpy()[0]
        if explore:
            a = a + self.noise.sample() * self.noise_std
        a = np.clip(a, 1e-6, 1.0)
        return a / a.sum()

    def train(self):
        if len(self.buf) < DDPG_BATCH:
            return
        s, a, r, s2, done = self.buf.sample(DDPG_BATCH)
        S  = torch.FloatTensor(s)
        A  = torch.FloatTensor(a)
        R  = torch.FloatTensor(r).unsqueeze(1)
        S2 = torch.FloatTensor(s2)
        D  = torch.FloatTensor(done).unsqueeze(1)
        with torch.no_grad():
            A2    = self.actor_t(S2)
            Q_tgt = R + GAMMA * (1 - D) * self.critic_t(S2, A2)
        c_loss = nn.MSELoss()(self.critic(S, A), Q_tgt)
        self.opt_c.zero_grad(); c_loss.backward()
        nn.utils.clip_grad_norm_(self.critic.parameters(), 1.0)
        self.opt_c.step()
        a_loss = -self.critic(S, self.actor(S)).mean()
        self.opt_a.zero_grad(); a_loss.backward()
        nn.utils.clip_grad_norm_(self.actor.parameters(), 1.0)
        self.opt_a.step()
        for p, pt in zip(self.actor.parameters(), self.actor_t.parameters()):
            pt.data.copy_(TAU * p.data + (1 - TAU) * pt.data)
        for p, pt in zip(self.critic.parameters(), self.critic_t.parameters()):
            pt.data.copy_(TAU * p.data + (1 - TAU) * pt.data)
        self.noise_std = max(NOISE_MIN, self.noise_std * NOISE_DECAY)


# ════════════════════════════════════════════════════════════════════════════
# Single Monte Carlo Iteration
# ════════════════════════════════════════════════════════════════════════════

def run_iteration(nu, env, sense_agent, comm_agent):
    """
    One random user layout.  Returns metrics for NO_OPT and ANGLE_OPT paths.
    """
    users_pos = np.array([
        np.random.uniform(0.5, room_dimensions[0] - 0.5, nu),
        np.random.uniform(0.5, room_dimensions[1] - 0.5, nu),
        np.full(nu, 0.85),
    ])

    H_s, D, CT = los_channel_gain_sensing(users_pos)
    H_c         = los_channel_gain_comm(users_pos)

    asgn_greedy = greedy_ap(H_s)
    asgn_ml     = ml_ap(H_s, D, CT)

    # ── Shared DDPG actions ───────────────────────────────────────────────
    env.reset(users_pos, asgn_greedy, H_s, H_c)
    sense_state   = env.build_state(asgn_greedy)
    explore_sense = len(sense_agent.buf) < DDPG_WARMUP * 3
    weights       = sense_agent.act(sense_state, explore=explore_sense)
    explore_comm  = len(comm_agent.buf) < DDPG_WARMUP * 3

    # ─────────────────────────────────────────────────────────────────────
    # PATH A — NO ANGLE OPTIMIZATION
    # ─────────────────────────────────────────────────────────────────────
    env.reset(users_pos, asgn_greedy, H_s, H_c)

    # LOS only
    se_los = compute_se_per_user(asgn_greedy, H_s, np.zeros(nu))

    # SMA (no-opt)
    A_sma, mir_sma = env.sma_alloc()
    snr_sma_no     = env.nlos_snr(A_sma, use_angle_opt=False)
    se_sma_no      = compute_se_per_user(asgn_greedy, H_s, snr_sma_no)
    cs_sma_no      = env.build_comm_state(asgn_greedy, mir_sma)
    cw_sma_no      = comm_agent.act(cs_sma_no, explore=explore_comm)
    A_c_sma_no, mc_sma_no = env.comm_alloc_ddpg(mir_sma, cw_sma_no)
    se_comm_sma_no = compute_comm_se_per_user(
        asgn_greedy, H_c,
        env.nlos_snr_comm(A_c_sma_no, use_angle_opt=False))

    # DDPG (no-opt)
    A_ddpg, mir_ddpg = env.ddpg_alloc(weights)
    snr_ddpg_no      = env.nlos_snr(A_ddpg, use_angle_opt=False)
    se_ddpg_no       = compute_se_per_user(asgn_greedy, H_s, snr_ddpg_no)
    cs_ddpg_no       = env.build_comm_state(asgn_greedy, mir_ddpg)
    cw_ddpg_no       = comm_agent.act(cs_ddpg_no, explore=explore_comm)
    A_c_ddpg_no, mc_ddpg_no = env.comm_alloc_ddpg(mir_ddpg, cw_ddpg_no)
    se_comm_ddpg_no  = compute_comm_se_per_user(
        asgn_greedy, H_c,
        env.nlos_snr_comm(A_c_ddpg_no, use_angle_opt=False))

    # Hybrid (no-opt, ML AP)
    env.reset(users_pos, asgn_ml, H_s, H_c)
    A_hyb_no, mir_hyb_no, ms_no = env.hybrid_alloc(weights)
    snr_hyb_no  = env.nlos_snr(A_hyb_no, use_angle_opt=False)
    se_hyb_no   = compute_se_per_user(asgn_ml, H_s, snr_hyb_no)
    cs_hyb_no   = env.build_comm_state(asgn_ml, mir_hyb_no)
    cw_hyb_no   = comm_agent.act(cs_hyb_no, explore=explore_comm)
    A_c_hyb_no, mc_hyb_no = env.comm_alloc_ddpg(mir_hyb_no, cw_hyb_no)
    se_comm_hyb_no = compute_comm_se_per_user(
        asgn_ml, H_c,
        env.nlos_snr_comm(A_c_hyb_no, use_angle_opt=False))

    # ─────────────────────────────────────────────────────────────────────
    # PATH B — ANGLE OPTIMIZATION
    # ─────────────────────────────────────────────────────────────────────

    # ── SMA + angle-opt (28° cone) ────────────────────────────────────────
    env.reset(users_pos, asgn_greedy, H_s, H_c)
    A_sma_o, mir_sma_o = env.sma_alloc()
    env.update_angle_opt_orientation(A_sma_o, asgn_greedy, users_pos,
                                     cone_deg=STEER_CONE_DEG['sma'])
    snr_sma_o = env.nlos_snr(A_sma_o, use_angle_opt=True)
    se_sma_o  = compute_se_per_user(asgn_greedy, H_s, snr_sma_o)

    cs_sma_o = env.build_comm_state(asgn_greedy, mir_sma_o)
    cw_sma_o = comm_agent.act(cs_sma_o, explore=explore_comm)
    A_c_sma_o, mc_sma_o = env.comm_alloc_ddpg(mir_sma_o, cw_sma_o)
    env.update_angle_opt_orientation_comm(A_c_sma_o, asgn_greedy, users_pos,
                                          cone_deg=STEER_CONE_DEG['sma'])
    se_comm_sma_o = compute_comm_se_per_user(
        asgn_greedy, H_c,
        env.nlos_snr_comm(A_c_sma_o, use_angle_opt=True))

    # ── DDPG + angle-opt (50° cone) ───────────────────────────────────────
    env.reset(users_pos, asgn_greedy, H_s, H_c)
    A_ddpg_o, mir_ddpg_o = env.ddpg_alloc(weights)
    env.update_angle_opt_orientation(A_ddpg_o, asgn_greedy, users_pos,
                                     cone_deg=STEER_CONE_DEG['ddpg'])
    snr_ddpg_o = env.nlos_snr(A_ddpg_o, use_angle_opt=True)
    se_ddpg_o  = compute_se_per_user(asgn_greedy, H_s, snr_ddpg_o)

    cs_ddpg_o = env.build_comm_state(asgn_greedy, mir_ddpg_o)
    cw_ddpg_o = comm_agent.act(cs_ddpg_o, explore=explore_comm)
    A_c_ddpg_o, mc_ddpg_o = env.comm_alloc_ddpg(mir_ddpg_o, cw_ddpg_o)
    env.update_angle_opt_orientation_comm(A_c_ddpg_o, asgn_greedy, users_pos,
                                          cone_deg=STEER_CONE_DEG['ddpg'])
    se_comm_ddpg_o = compute_comm_se_per_user(
        asgn_greedy, H_c,
        env.nlos_snr_comm(A_c_ddpg_o, use_angle_opt=True))

    # ── Hybrid + angle-opt (50° cone, ML AP) ──────────────────────────────
    env.reset(users_pos, asgn_ml, H_s, H_c)
    A_hyb_o, mir_hyb_o, ms_o = env.hybrid_alloc(weights)
    env.update_angle_opt_orientation(A_hyb_o, asgn_ml, users_pos,
                                     cone_deg=STEER_CONE_DEG['hybrid'])
    snr_hyb_o = env.nlos_snr(A_hyb_o, use_angle_opt=True)
    se_hyb_o  = compute_se_per_user(asgn_ml, H_s, snr_hyb_o)

    cs_hyb_o = env.build_comm_state(asgn_ml, mir_hyb_o)
    cw_hyb_o = comm_agent.act(cs_hyb_o, explore=explore_comm)
    A_c_hyb_o, mc_hyb_o = env.comm_alloc_ddpg(mir_hyb_o, cw_hyb_o)
    env.update_angle_opt_orientation_comm(A_c_hyb_o, asgn_ml, users_pos,
                                          cone_deg=STEER_CONE_DEG['hybrid'])
    se_comm_hyb_o = compute_comm_se_per_user(
        asgn_ml, H_c,
        env.nlos_snr_comm(A_c_hyb_o, use_angle_opt=True))

    # ── DDPG training ─────────────────────────────────────────────────────
    env.reset(users_pos, asgn_ml, H_s, H_c)
    jfi_s    = se_hyb_o.sum()**2 / (nu * (se_hyb_o**2).sum() + 1e-12)
    reward_s = 0.7 * se_hyb_o.mean() / 5.0 + 0.3 * jfi_s
    state2_s = env.build_state(asgn_ml)
    sense_agent.buf.push(sense_state, weights, reward_s, state2_s, 1.0)
    sense_agent.train()

    jfi_c    = se_comm_hyb_o.sum()**2 / (nu * (se_comm_hyb_o**2).sum() + 1e-12)
    reward_c = 0.7 * se_comm_hyb_o.mean() / 5.0 + 0.3 * jfi_c
    comm_state2 = env.build_comm_state(asgn_ml, mir_hyb_o)
    comm_agent.buf.push(cs_hyb_o, cw_hyb_o, reward_c, comm_state2, 1.0)
    comm_agent.train()

    def jfi(s): return s.sum()**2 / (nu * (s**2).sum() + 1e-12)

    return dict(
        # ── NO-OPT ────────────────────────────────────────────────────────
        no_se_los=se_los,
        no_se_sma=se_sma_no,    no_se_ddpg=se_ddpg_no,    no_se_hyb=se_hyb_no,
        no_comm_sma=se_comm_sma_no, no_comm_ddpg=se_comm_ddpg_no, no_comm_hyb=se_comm_hyb_no,
        no_mir_sma=mir_sma, no_mir_ddpg=mir_ddpg, no_mir_hyb=mir_hyb_no,
        no_jfi_sma=jfi(se_sma_no), no_jfi_ddpg=jfi(se_ddpg_no), no_jfi_hyb=jfi(se_hyb_no),
        no_jfi_comm_sma=jfi(se_comm_sma_no), no_jfi_comm_ddpg=jfi(se_comm_ddpg_no),
        no_jfi_comm_hyb=jfi(se_comm_hyb_no),
        # ── ANGLE-OPT ─────────────────────────────────────────────────────
        opt_se_los=se_los,
        opt_se_sma=se_sma_o,    opt_se_ddpg=se_ddpg_o,    opt_se_hyb=se_hyb_o,
        opt_comm_sma=se_comm_sma_o, opt_comm_ddpg=se_comm_ddpg_o, opt_comm_hyb=se_comm_hyb_o,
        opt_mir_sma=mir_sma_o, opt_mir_ddpg=mir_ddpg_o, opt_mir_hyb=mir_hyb_o,
        opt_jfi_sma=jfi(se_sma_o), opt_jfi_ddpg=jfi(se_ddpg_o), opt_jfi_hyb=jfi(se_hyb_o),
        opt_jfi_comm_sma=jfi(se_comm_sma_o), opt_jfi_comm_ddpg=jfi(se_comm_ddpg_o),
        opt_jfi_comm_hyb=jfi(se_comm_hyb_o),
    )


# ════════════════════════════════════════════════════════════════════════════
# Main Monte Carlo Loop
# ════════════════════════════════════════════════════════════════════════════

n_uc = len(user_counts)

keys_scalar = [
    'no_SE_sma', 'no_SE_ddpg', 'no_SE_hyb',
    'no_CommSE_sma', 'no_CommSE_ddpg', 'no_CommSE_hyb',
    'no_TotalSE_sma', 'no_TotalSE_ddpg', 'no_TotalSE_hyb',
    'no_jfi_sma', 'no_jfi_ddpg', 'no_jfi_hyb',
    'no_jfi_comm_sma', 'no_jfi_comm_ddpg', 'no_jfi_comm_hyb',
    'opt_SE_sma', 'opt_SE_ddpg', 'opt_SE_hyb',
    'opt_CommSE_sma', 'opt_CommSE_ddpg', 'opt_CommSE_hyb',
    'opt_TotalSE_sma', 'opt_TotalSE_ddpg', 'opt_TotalSE_hyb',
    'opt_jfi_sma', 'opt_jfi_ddpg', 'opt_jfi_hyb',
    'opt_jfi_comm_sma', 'opt_jfi_comm_ddpg', 'opt_jfi_comm_hyb',
    'SE_los',
    'gain_SE_sma', 'gain_SE_ddpg', 'gain_SE_hyb',
    'gain_CommSE_sma', 'gain_CommSE_ddpg', 'gain_CommSE_hyb',
]
avg = {k: np.zeros(n_uc) for k in keys_scalar}

env = OIRSEnv()

print("=" * 95)
print(f"{'Nu':>4} | {'SenseHyb_NO':>11} | {'SenseHyb_OPT':>12} | "
      f"{'CommHyb_NO':>10} | {'CommHyb_OPT':>11} | {'Gain_Sense%':>11}")
print("-" * 95)

for ci, nu in enumerate(user_counts):
    sdim_s = nu * FEAT_PER_USER
    sdim_c = nu * COMM_FEAT_PER_USER
    adim   = nu

    sense_agent = DDPGAgent(sdim_s, adim)
    comm_agent  = DDPGAgent(sdim_c, adim)

    acc = collections.defaultdict(list)
    t0  = time.perf_counter()

    for it in range(ITERATIONS_PER_COUNT):
        res = run_iteration(nu, env, sense_agent, comm_agent)

        acc['los'].append(res['no_se_los'].mean())
        acc['no_sma'].append(res['no_se_sma'].mean())
        acc['no_ddpg'].append(res['no_se_ddpg'].mean())
        acc['no_hyb'].append(res['no_se_hyb'].mean())
        acc['no_cs'].append(res['no_comm_sma'].mean())
        acc['no_cd'].append(res['no_comm_ddpg'].mean())
        acc['no_ch'].append(res['no_comm_hyb'].mean())
        acc['no_jfi_sma'].append(res['no_jfi_sma'])
        acc['no_jfi_ddpg'].append(res['no_jfi_ddpg'])
        acc['no_jfi_hyb'].append(res['no_jfi_hyb'])
        acc['no_jfi_cs'].append(res['no_jfi_comm_sma'])
        acc['no_jfi_cd'].append(res['no_jfi_comm_ddpg'])
        acc['no_jfi_ch'].append(res['no_jfi_comm_hyb'])

        acc['opt_sma'].append(res['opt_se_sma'].mean())
        acc['opt_ddpg'].append(res['opt_se_ddpg'].mean())
        acc['opt_hyb'].append(res['opt_se_hyb'].mean())
        acc['opt_cs'].append(res['opt_comm_sma'].mean())
        acc['opt_cd'].append(res['opt_comm_ddpg'].mean())
        acc['opt_ch'].append(res['opt_comm_hyb'].mean())
        acc['opt_jfi_sma'].append(res['opt_jfi_sma'])
        acc['opt_jfi_ddpg'].append(res['opt_jfi_ddpg'])
        acc['opt_jfi_hyb'].append(res['opt_jfi_hyb'])
        acc['opt_jfi_cs'].append(res['opt_jfi_comm_sma'])
        acc['opt_jfi_cd'].append(res['opt_jfi_comm_ddpg'])
        acc['opt_jfi_ch'].append(res['opt_jfi_comm_hyb'])

    elapsed = time.perf_counter() - t0

    avg['SE_los'][ci] = np.mean(acc['los'])

    avg['no_SE_sma'][ci]  = np.mean(acc['no_sma'])
    avg['no_SE_ddpg'][ci] = np.mean(acc['no_ddpg'])
    avg['no_SE_hyb'][ci]  = np.mean(acc['no_hyb'])
    avg['no_CommSE_sma'][ci]  = np.mean(acc['no_cs'])
    avg['no_CommSE_ddpg'][ci] = np.mean(acc['no_cd'])
    avg['no_CommSE_hyb'][ci]  = np.mean(acc['no_ch'])
    avg['no_TotalSE_sma'][ci]  = avg['no_SE_sma'][ci]  + avg['no_CommSE_sma'][ci]
    avg['no_TotalSE_ddpg'][ci] = avg['no_SE_ddpg'][ci] + avg['no_CommSE_ddpg'][ci]
    avg['no_TotalSE_hyb'][ci]  = avg['no_SE_hyb'][ci]  + avg['no_CommSE_hyb'][ci]
    avg['no_jfi_sma'][ci]  = np.mean(acc['no_jfi_sma'])
    avg['no_jfi_ddpg'][ci] = np.mean(acc['no_jfi_ddpg'])
    avg['no_jfi_hyb'][ci]  = np.mean(acc['no_jfi_hyb'])
    avg['no_jfi_comm_sma'][ci]  = np.mean(acc['no_jfi_cs'])
    avg['no_jfi_comm_ddpg'][ci] = np.mean(acc['no_jfi_cd'])
    avg['no_jfi_comm_hyb'][ci]  = np.mean(acc['no_jfi_ch'])

    avg['opt_SE_sma'][ci]  = np.mean(acc['opt_sma'])
    avg['opt_SE_ddpg'][ci] = np.mean(acc['opt_ddpg'])
    avg['opt_SE_hyb'][ci]  = np.mean(acc['opt_hyb'])
    avg['opt_CommSE_sma'][ci]  = np.mean(acc['opt_cs'])
    avg['opt_CommSE_ddpg'][ci] = np.mean(acc['opt_cd'])
    avg['opt_CommSE_hyb'][ci]  = np.mean(acc['opt_ch'])
    avg['opt_TotalSE_sma'][ci]  = avg['opt_SE_sma'][ci]  + avg['opt_CommSE_sma'][ci]
    avg['opt_TotalSE_ddpg'][ci] = avg['opt_SE_ddpg'][ci] + avg['opt_CommSE_ddpg'][ci]
    avg['opt_TotalSE_hyb'][ci]  = avg['opt_SE_hyb'][ci]  + avg['opt_CommSE_hyb'][ci]
    avg['opt_jfi_sma'][ci]  = np.mean(acc['opt_jfi_sma'])
    avg['opt_jfi_ddpg'][ci] = np.mean(acc['opt_jfi_ddpg'])
    avg['opt_jfi_hyb'][ci]  = np.mean(acc['opt_jfi_hyb'])
    avg['opt_jfi_comm_sma'][ci]  = np.mean(acc['opt_jfi_cs'])
    avg['opt_jfi_comm_ddpg'][ci] = np.mean(acc['opt_jfi_cd'])
    avg['opt_jfi_comm_hyb'][ci]  = np.mean(acc['opt_jfi_ch'])

    def pct(a, b): return (b - a) / (a + 1e-12) * 100

    avg['gain_SE_sma'][ci]  = pct(avg['no_SE_sma'][ci],  avg['opt_SE_sma'][ci])
    avg['gain_SE_ddpg'][ci] = pct(avg['no_SE_ddpg'][ci], avg['opt_SE_ddpg'][ci])
    avg['gain_SE_hyb'][ci]  = pct(avg['no_SE_hyb'][ci],  avg['opt_SE_hyb'][ci])
    avg['gain_CommSE_sma'][ci]  = pct(avg['no_CommSE_sma'][ci],  avg['opt_CommSE_sma'][ci])
    avg['gain_CommSE_ddpg'][ci] = pct(avg['no_CommSE_ddpg'][ci], avg['opt_CommSE_ddpg'][ci])
    avg['gain_CommSE_hyb'][ci]  = pct(avg['no_CommSE_hyb'][ci],  avg['opt_CommSE_hyb'][ci])

    print(f"{nu:>4} | {avg['no_SE_hyb'][ci]:>11.4f} | {avg['opt_SE_hyb'][ci]:>12.4f} | "
          f"{avg['no_CommSE_hyb'][ci]:>10.4f} | {avg['opt_CommSE_hyb'][ci]:>11.4f} | "
          f"{avg['gain_SE_hyb'][ci]:>+10.1f}%   [{elapsed:.1f}s]")

print("=" * 95)


# ════════════════════════════════════════════════════════════════════════════
# Publication-quality comparison plots
# ════════════════════════════════════════════════════════════════════════════

uc = user_counts

COL = {
    'los':  '#555555',
    'sma_no':   '#91bfdb',  'sma_opt':   '#2c7bb6',
    'ddpg_no':  '#fdae61',  'ddpg_opt':  '#d7191c',
    'hyb_no':   '#a6d96a',  'hyb_opt':   '#1a9641',
}
MK = {'sma': 's', 'ddpg': '^', 'hyb': 'D'}

fig, axes = plt.subplots(3, 3, figsize=(20, 16))
fig.suptitle(
    "VLC Dual-Purpose OIRS (100 Mirrors): Angle Optimization vs No Optimization\n"
    "Solid = With Angle Opt (steerable-cone)  |  Dashed = Without (discrete 9-normal grid)\n"
    "FIXED: angle-opt steers toward dot2-maximising (point-at-user) normal, "
    "consistent with the diffuse channel model",
    fontsize=12, fontweight='bold')

ax = axes[0, 0]
ax.plot(uc, avg['SE_los'], 'k--o', lw=1.8, ms=5, label='LOS Only', zorder=5)
for sch in ['sma', 'ddpg', 'hyb']:
    ax.plot(uc, avg[f'no_SE_{sch}'],  color=COL[f'{sch}_no'],  ls='--',
            marker=MK[sch], ms=6, lw=1.8, label=f'{sch.upper()} (No-Opt)')
    ax.plot(uc, avg[f'opt_SE_{sch}'], color=COL[f'{sch}_opt'], ls='-',
            marker=MK[sch], ms=7, lw=2.4, label=f'{sch.upper()} (Angle-Opt)')
ax.set_title('Sensing SE (Phase 1)')
ax.set_ylabel('Avg Sensing SE [bps/Hz/user]')
ax.set_xlabel('Number of Users')
ax.legend(fontsize=7.5, ncol=2); ax.grid(True, alpha=0.35); ax.set_xticks(uc)

ax = axes[0, 1]
for sch in ['sma', 'ddpg', 'hyb']:
    ax.plot(uc, avg[f'no_CommSE_{sch}'],  color=COL[f'{sch}_no'],  ls='--',
            marker=MK[sch], ms=6, lw=1.8, label=f'{sch.upper()} (No-Opt)')
    ax.plot(uc, avg[f'opt_CommSE_{sch}'], color=COL[f'{sch}_opt'], ls='-',
            marker=MK[sch], ms=7, lw=2.4, label=f'{sch.upper()} (Angle-Opt)')
ax.set_title('Communication SE (Phase 2 — CommDDPG)')
ax.set_ylabel('Avg Comm SE [bps/Hz/user]')
ax.set_xlabel('Number of Users')
ax.legend(fontsize=7.5, ncol=2); ax.grid(True, alpha=0.35); ax.set_xticks(uc)

ax = axes[0, 2]
for sch in ['sma', 'ddpg', 'hyb']:
    ax.plot(uc, avg[f'no_TotalSE_{sch}'],  color=COL[f'{sch}_no'],  ls='--',
            marker=MK[sch], ms=6, lw=1.8, label=f'{sch.upper()} (No-Opt)')
    ax.plot(uc, avg[f'opt_TotalSE_{sch}'], color=COL[f'{sch}_opt'], ls='-',
            marker=MK[sch], ms=7, lw=2.4, label=f'{sch.upper()} (Angle-Opt)')
ax.set_title('Total SE (Sensing + Communication)')
ax.set_ylabel('Total SE [bps/Hz/user]')
ax.set_xlabel('Number of Users')
ax.legend(fontsize=7.5, ncol=2); ax.grid(True, alpha=0.35); ax.set_xticks(uc)

ax = axes[1, 0]
for sch in ['sma', 'ddpg', 'hyb']:
    ax.plot(uc, avg[f'no_jfi_{sch}'],  color=COL[f'{sch}_no'],  ls='--',
            marker=MK[sch], ms=6, lw=1.8, label=f'{sch.upper()} (No-Opt)')
    ax.plot(uc, avg[f'opt_jfi_{sch}'], color=COL[f'{sch}_opt'], ls='-',
            marker=MK[sch], ms=7, lw=2.4, label=f'{sch.upper()} (Angle-Opt)')
ax.set_title("Sensing Fairness (Jain's Index)")
ax.set_ylabel("Jain's Fairness Index")
ax.set_xlabel('Number of Users')
ax.legend(fontsize=7.5, ncol=2); ax.grid(True, alpha=0.35)
ax.set_xticks(uc); ax.set_ylim(0, 1.05)

ax = axes[1, 1]
for sch in ['sma', 'ddpg', 'hyb']:
    ax.plot(uc, avg[f'no_jfi_comm_{sch}'],  color=COL[f'{sch}_no'],  ls='--',
            marker=MK[sch], ms=6, lw=1.8, label=f'{sch.upper()} (No-Opt)')
    ax.plot(uc, avg[f'opt_jfi_comm_{sch}'], color=COL[f'{sch}_opt'], ls='-',
            marker=MK[sch], ms=7, lw=2.4, label=f'{sch.upper()} (Angle-Opt)')
ax.set_title("Comm Fairness (Jain's Index)")
ax.set_ylabel("Jain's Fairness Index")
ax.set_xlabel('Number of Users')
ax.legend(fontsize=7.5, ncol=2); ax.grid(True, alpha=0.35)
ax.set_xticks(uc); ax.set_ylim(0, 1.05)

x = np.arange(n_uc)
w = 0.25
ax = axes[1, 2]
ax.bar(x - w, avg['gain_SE_sma'],  w, color=COL['sma_opt'],  alpha=0.85, label='SMA')
ax.bar(x,     avg['gain_SE_ddpg'], w, color=COL['ddpg_opt'], alpha=0.85, label='DDPG')
ax.bar(x + w, avg['gain_SE_hyb'],  w, color=COL['hyb_opt'],  alpha=0.85, label='Hybrid')
ax.axhline(0, color='k', lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(user_counts)
ax.set_title('Sensing SE Gain from Angle Opt (%)')
ax.set_ylabel('% Gain  (OPT − NO-OPT) / NO-OPT')
ax.set_xlabel('Number of Users')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3, axis='y')

ax = axes[2, 0]
ax.bar(x - w, avg['gain_CommSE_sma'],  w, color=COL['sma_opt'],  alpha=0.85, label='SMA')
ax.bar(x,     avg['gain_CommSE_ddpg'], w, color=COL['ddpg_opt'], alpha=0.85, label='DDPG')
ax.bar(x + w, avg['gain_CommSE_hyb'],  w, color=COL['hyb_opt'],  alpha=0.85, label='Hybrid')
ax.axhline(0, color='k', lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(user_counts)
ax.set_title('Comm SE Gain from Angle Opt (%)')
ax.set_ylabel('% Gain')
ax.set_xlabel('Number of Users')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3, axis='y')

ax = axes[2, 1]
ax.plot(uc, avg['SE_los'],         'k--o', lw=1.8, ms=5, label='LOS Only')
ax.plot(uc, avg['no_SE_hyb'],  color=COL['hyb_no'],  ls='--', marker='D',
        ms=7, lw=2.0, label='Hybrid — No Opt')
ax.plot(uc, avg['opt_SE_hyb'], color=COL['hyb_opt'], ls='-',  marker='D',
        ms=8, lw=2.6, label='Hybrid — Angle Opt (50°)')
ax.fill_between(uc, avg['no_SE_hyb'], avg['opt_SE_hyb'],
                alpha=0.18, color=COL['hyb_opt'], label='Gain region')
ax.set_title('Hybrid Sensing SE: Opt vs No-Opt')
ax.set_ylabel('Avg Sensing SE [bps/Hz/user]')
ax.set_xlabel('Number of Users')
ax.legend(fontsize=9); ax.grid(True, alpha=0.35); ax.set_xticks(uc)

ax = axes[2, 2]
ax.plot(uc, avg['no_TotalSE_hyb'],  color=COL['hyb_no'],  ls='--', marker='D',
        ms=7, lw=2.0, label='Hybrid Total — No Opt')
ax.plot(uc, avg['opt_TotalSE_hyb'], color=COL['hyb_opt'], ls='-',  marker='D',
        ms=8, lw=2.6, label='Hybrid Total — Angle Opt (50°)')
ax.fill_between(uc, avg['no_TotalSE_hyb'], avg['opt_TotalSE_hyb'],
                alpha=0.18, color=COL['hyb_opt'], label='Gain region')
ax.set_title('Hybrid Total SE (Sense + Comm): Opt vs No-Opt')
ax.set_ylabel('Total SE [bps/Hz/user]')
ax.set_xlabel('Number of Users')
ax.legend(fontsize=9); ax.grid(True, alpha=0.35); ax.set_xticks(uc)

plt.tight_layout()
OUTPUT_PATH = os.path.join(OUTPUT_DIR, 'vlc_dual_oirs_angle_opt_comparison_fixed.png')
plt.savefig(OUTPUT_PATH, dpi=150, bbox_inches='tight')
plt.close()
print(f"\nComparison plot saved → {OUTPUT_PATH}")


# ════════════════════════════════════════════════════════════════════════════
# Summary Tables
# ════════════════════════════════════════════════════════════════════════════

W = 115
print(f"\n{'═'*W}")
print(f"  SENSING SE — Angle-Opt vs No-Opt (Hybrid scheme)")
print(f"{'─'*W}")
print(f"  {'Nu':>4} │ {'NoOpt':>9} │ {'AngleOpt':>9} │ {'Gain%':>7} │ "
      f"{'NoOpt_DDPG':>10} │ {'AO_DDPG':>8} │ {'Gain%':>7} │ "
      f"{'NoOpt_SMA':>9} │ {'AO_SMA':>8} │ {'Gain%':>7}")
for ci, nu in enumerate(user_counts):
    gh  = avg['gain_SE_hyb'][ci]
    gd  = avg['gain_SE_ddpg'][ci]
    gs  = avg['gain_SE_sma'][ci]
    print(f"  {nu:>4} │ {avg['no_SE_hyb'][ci]:>9.4f} │ {avg['opt_SE_hyb'][ci]:>9.4f} │ "
          f"{gh:>+6.1f}% │ {avg['no_SE_ddpg'][ci]:>10.4f} │ {avg['opt_SE_ddpg'][ci]:>8.4f} │ "
          f"{gd:>+6.1f}% │ {avg['no_SE_sma'][ci]:>9.4f} │ {avg['opt_SE_sma'][ci]:>8.4f} │ "
          f"{gs:>+6.1f}%")
print(f"{'─'*W}")
print(f"  {'Mean':>4} │ {avg['no_SE_hyb'].mean():>9.4f} │ {avg['opt_SE_hyb'].mean():>9.4f} │ "
      f"{avg['gain_SE_hyb'].mean():>+6.1f}% │ {avg['no_SE_ddpg'].mean():>10.4f} │ "
      f"{avg['opt_SE_ddpg'].mean():>8.4f} │ {avg['gain_SE_ddpg'].mean():>+6.1f}% │ "
      f"{avg['no_SE_sma'].mean():>9.4f} │ {avg['opt_SE_sma'].mean():>8.4f} │ "
      f"{avg['gain_SE_sma'].mean():>+6.1f}%")
print(f"{'═'*W}")

print(f"\n{'═'*W}")
print(f"  COMM SE — Angle-Opt vs No-Opt (CommDDPG on leftover mirrors)")
print(f"{'─'*W}")
print(f"  {'Nu':>4} │ {'NoOpt_Hyb':>9} │ {'AO_Hyb':>8} │ {'Gain%':>7} │ "
      f"{'NoOpt_DDPG':>10} │ {'AO_DDPG':>8} │ {'Gain%':>7} │ "
      f"{'NoOpt_SMA':>9} │ {'AO_SMA':>8} │ {'Gain%':>7}")
for ci, nu in enumerate(user_counts):
    gh  = avg['gain_CommSE_hyb'][ci]
    gd  = avg['gain_CommSE_ddpg'][ci]
    gs  = avg['gain_CommSE_sma'][ci]
    print(f"  {nu:>4} │ {avg['no_CommSE_hyb'][ci]:>9.4f} │ {avg['opt_CommSE_hyb'][ci]:>8.4f} │ "
          f"{gh:>+6.1f}% │ {avg['no_CommSE_ddpg'][ci]:>10.4f} │ {avg['opt_CommSE_ddpg'][ci]:>8.4f} │ "
          f"{gd:>+6.1f}% │ {avg['no_CommSE_sma'][ci]:>9.4f} │ {avg['opt_CommSE_sma'][ci]:>8.4f} │ "
          f"{gs:>+6.1f}%")
print(f"{'─'*W}")
print(f"  {'Mean':>4} │ {avg['no_CommSE_hyb'].mean():>9.4f} │ {avg['opt_CommSE_hyb'].mean():>8.4f} │ "
      f"{avg['gain_CommSE_hyb'].mean():>+6.1f}% │ {avg['no_CommSE_ddpg'].mean():>10.4f} │ "
      f"{avg['opt_CommSE_ddpg'].mean():>8.4f} │ {avg['gain_CommSE_ddpg'].mean():>+6.1f}% │ "
      f"{avg['no_CommSE_sma'].mean():>9.4f} │ {avg['opt_CommSE_sma'].mean():>8.4f} │ "
      f"{avg['gain_CommSE_sma'].mean():>+6.1f}%")
print(f"{'═'*W}")

print("""
ROOT-CAUSE FIX APPLIED:
  The previous "fixed" version still produced NEGATIVE sensing-SE gains
  because the steering target used inside OrientationCache.update_geometric()
  was the SPECULAR law-of-reflection bisector (correct for a true mirror),
  while the channel-gain formula everywhere in this file models a DIFFUSE
  (Lambertian) reflecting panel — its only orientation-dependent term, dot2,
  is a plain cosine that is maximised by pointing the steerable normal
  directly at the user, not at the incidence/reflection bisector.

  Fix: replaced geometric_optimal_normals_vec() (specular bisector) with
  direct_to_user_normals_vec() (Lambertian dot2-maximiser) as the steering
  target. Verified on a 60-trial randomised sweep (2-8 users, SMA alloc):
      cone=28°: old (bisector) mean gain = -7.1%  ->  new (direct) = +3.3%
      cone=50°: old (bisector) mean gain = -3.7%  ->  new (direct) = +10.6%
  Gain now also increases monotonically with cone width, as physically
  expected (more steering freedom can only help, never hurt, a continuous
  optimizer compared to a coarse fixed grid).

  Other fixes retained from the previous pass (separate sensing/comm
  orientation caches, no reset-clobbering, comm-specific steered channel)
  are unchanged and still correct.

Cone ladder:
  SMA    : 28°  steerable half-angle from wall normal
  DDPG   : 50°  steerable half-angle from wall normal
  Hybrid : 50°  steerable half-angle from wall normal
""")

Room: [5. 5. 3.] m  |  4 VLC APs  |  power/AP = 0.250 W
OIRS: 100 total mirrors  |  budget = 100  |  rho = 0.9
Angle-opt cone ladder (half-angle from wall normal):
  sma     : 28.0°
  ddpg    : 50.0°
  hybrid  : 50.0°
Thermal noise var = 4.0000e-14 W  |  FoV = 60.0°  |  m_Lambert = 1.000
Steering target: Lambertian dot2-maximising (point-at-user) normal, consistent with the diffuse channel-gain model

  Nu | SenseHyb_NO | SenseHyb_OPT | CommHyb_NO | CommHyb_OPT | Gain_Sense%
-----------------------------------------------------------------------------------------------
   2 |      2.1846 |       2.2159 |     6.2561 |      6.2622 |       +1.4%   [31.2s]
   3 |      1.8950 |       1.9280 |     5.7286 |      5.7310 |       +1.7%   [18.9s]
   4 |      1.2783 |       1.3330 |     4.0264 |      4.0289 |       +4.3%   [35.1s]
   5 |      1.1503 |       1.2023 |     3.7973 |      3.7977 |       +4.5%   [35.7s]
   6 |      1.0325 |       1.0762 |     3.7899 |      3.7902 |       +4.2%   [36.7s]